In [0]:
bronze_accounts_df = spark.table(
    "banking_catalog.banking_schema.bronze_accounts"
)

In [0]:
bronze_accounts_df.count()

518581

In [0]:
bronze_accounts_df.groupBy(
    bronze_accounts_df.columns
).count().filter("count > 1").count()

0

In [0]:
from pyspark.sql.functions import *

bronze_accounts_df.select([
    sum(when(col(c).isNull(),1).otherwise(0)).alias(c)
    for c in bronze_accounts_df.columns
]).show()

+---------+-------+--------------+---------+-----------+
|bank_name|bank_id|account_number|entity_id|entity_name|
+---------+-------+--------------+---------+-----------+
|        0|      0|             0|        0|          0|
+---------+-------+--------------+---------+-----------+



In [0]:
from pyspark.sql.functions import *

silver_accounts_df = (
    bronze_accounts_df
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "load_date",
        current_date()
    )
)

In [0]:
silver_accounts_df.count()

518581

In [0]:
silver_accounts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "banking_catalog.banking_schema.silver_accounts"
    )

In [0]:
spark.sql("""
SELECT COUNT(*)
FROM banking_catalog.banking_schema.silver_accounts
""").show()

+--------+
|COUNT(*)|
+--------+
|  518581|
+--------+

